In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from dotenv import load_dotenv

# Configurações visuais
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Carregar variáveis de ambiente
load_dotenv()

# Conexão MongoDB
MONGO_URI = os.getenv("MONGODB_URI")
mongo_client = MongoClient(MONGO_URI)
db = mongo_client["experiments_db"]
llm_judge_collection = db["llm_judge_comparative_metrics"]

# Métricas avaliadas
METRICS = ["coherence", "specificity", "informativeness", "relevance", "Understandability"]
METRIC_LABELS = {
    "coherence": "Coerência",
    "specificity": "Especificidade",
    "informativeness": "Informatividade",
    "relevance": "Relevância",
    "Understandability": "Compreensibilidade"
}

print("✅ Bibliotecas importadas e conexão MongoDB estabelecida!")

✅ Bibliotecas importadas e conexão MongoDB estabelecida!


## 2. Carregamento e Processamento dos Dados

In [2]:
# Carregar todos os documentos da collection
documents = list(llm_judge_collection.find())

print(f"📊 Total de experimentos encontrados: {len(documents)}\n")

# Processar dados
models = []

for doc in documents:
    experiment_name = doc["experiment_name"]
    # Extrair nome do modelo (remover prefixo "LLM_Judge_")
    model_name = experiment_name.replace("LLM_Judge_", "")
    models.append(model_name)
    
print("✅ Dados processados com sucesso!")
print(f"\n📋 Modelos encontrados:")
for model in models:
    print(f"   • {model}")

📊 Total de experimentos encontrados: 4

✅ Dados processados com sucesso!

📋 Modelos encontrados:
   • gpt-4o-mini_TEST
   • gpt-4o-mini
   • llama-3.3-70b-versatile
   • gpt-3.5-turbo-0125


## 3. Tabela Detalhada - Abordagem MTI

In [3]:
# Criar tabela detalhada para MTI
mti_table_rows = []

for doc in documents:
    experiment_name = doc["experiment_name"]
    model_name = experiment_name.replace("LLM_Judge_", "")
    
    row_data = {"Modelo": model_name}
    
    # MTI metrics
    mti_metrics = doc.get("MTI_metrics", {})
    
    # Para cada métrica individual
    for metric in METRICS:
        if metric in mti_metrics:
            mean_val = mti_metrics[metric].get("mean", 0)
            count_max = mti_metrics[metric].get("count_max_score", 0)
            total = mti_metrics[metric].get("total_samples", 1)
            
            # Calcular porcentagem de pontuação máxima
            pct_max = (count_max / total * 100) if total > 0 else 0
            
            # Formatar: "μ: 4.57 | Max: 57.71%"
            col_value = f"μ: {mean_val:.2f} | Max: {pct_max:.2f}%"
        else:
            col_value = "N/A"
        
        # Usar o label da métrica
        metric_label = METRIC_LABELS[metric]
        row_data[metric_label] = col_value
    
    # Adicionar coluna de média geral
    overall = doc.get("overall_statistics", {}).get("MTI", {})
    overall_mean = overall.get("mean", 0)
    overall_count_max = overall.get("count_max_score", 0)
    overall_total = overall.get("total_samples", 1)
    overall_pct_max = (overall_count_max / overall_total * 100) if overall_total > 0 else 0
    
    row_data["Média Geral"] = f"μ: {overall_mean:.2f} | Max: {overall_pct_max:.2f}%"
    
    mti_table_rows.append(row_data)

df_mti_detailed = pd.DataFrame(mti_table_rows)

print("\n" + "="*150)
print("📊 TABELA DETALHADA - ABORDAGEM MTI")
print("="*150)
print("Formato: μ: [Média] | Max: [% Pontuação Máxima]")
print("-"*150)
display(df_mti_detailed)


📊 TABELA DETALHADA - ABORDAGEM MTI
Formato: μ: [Média] | Max: [% Pontuação Máxima]
------------------------------------------------------------------------------------------------------------------------------------------------------


,Modelo,Coerência,Especificidade,Informatividade,Relevância,Compreensibilidade,Média Geral
0,gpt-4o-mini_TEST,μ: 4.40 | Max: 40.00%,μ: 3.60 | Max: 0.00%,μ: 4.00 | Max: 0.00%,μ: 5.00 | Max: 100.00%,μ: 4.40 | Max: 40.00%,μ: 4.28 | Max: 36.00%
1,gpt-4o-mini,μ: 4.63 | Max: 63.92%,μ: 4.37 | Max: 53.08%,μ: 4.55 | Max: 57.67%,μ: 4.96 | Max: 95.96%,μ: 4.71 | Max: 71.67%,μ: 4.64 | Max: 68.46%
2,llama-3.3-70b-versatile,μ: 4.52 | Max: 53.87%,μ: 4.28 | Max: 44.50%,μ: 4.46 | Max: 49.54%,μ: 4.96 | Max: 95.67%,μ: 4.65 | Max: 65.96%,μ: 4.57 | Max: 61.91%
3,gpt-3.5-turbo-0125,μ: 4.34 | Max: 45.88%,μ: 3.96 | Max: 35.79%,μ: 4.14 | Max: 39.38%,μ: 4.79 | Max: 84.96%,μ: 4.47 | Max: 55.50%,μ: 4.34 | Max: 52.30%


## 4. Tabela Detalhada - Abordagem STI

In [ ]:
# Criar tabela detalhada para STI
sti_table_rows = []

for doc in documents:
    experiment_name = doc["experiment_name"]
    model_name = experiment_name.replace("LLM_Judge_", "")
    
    row_data = {"Modelo": model_name}
    
    # STI metrics
    sti_metrics = doc.get("STI_metrics", {})
    
    # Para cada métrica individual
    for metric in METRICS:
        if metric in sti_metrics:
            mean_val = sti_metrics[metric].get("mean", 0)
            count_max = sti_metrics[metric].get("count_max_score", 0)
            total = sti_metrics[metric].get("total_samples", 1)
            
            # Calcular porcentagem de pontuação máxima
            pct_max = (count_max / total * 100) if total > 0 else 0
            
            # Formatar: "μ: 4.57 | Max: 57.71%"
            col_value = f"μ: {mean_val:.2f} | Max: {pct_max:.2f}%"
        else:
            col_value = "N/A"
        
        # Usar o label da métrica
        metric_label = METRIC_LABELS[metric]
        row_data[metric_label] = col_value
    
    # Adicionar coluna de média geral
    overall = doc.get("overall_statistics", {}).get("STI", {})
    overall_mean = overall.get("mean", 0)
    overall_count_max = overall.get("count_max_score", 0)
    overall_total = overall.get("total_samples", 1)
    overall_pct_max = (overall_count_max / overall_total * 100) if overall_total > 0 else 0
    
    row_data["Média Geral"] = f"μ: {overall_mean:.2f} | Max: {overall_pct_max:.2f}%"
    
    sti_table_rows.append(row_data)

df_sti_detailed = pd.DataFrame(sti_table_rows)

print("\n" + "="*150)
print("📊 TABELA DETALHADA - ABORDAGEM STI")
print("="*150)
print("Formato: μ: [Média] | Max: [% Pontuação Máxima]")
print("-"*150)
display(df_sti_detailed)

## 5. Exportação das Tabelas para LaTeX e PNG

In [ ]:
import dataframe_image as dfi
from datetime import datetime

# Criar diretório para exportação se não existir
export_dir = "inference/final_metrics/exports"
os.makedirs(export_dir, exist_ok=True)

# Timestamp para nomes de arquivo únicos
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("🚀 Iniciando exportação das tabelas...")
print("="*100)

### 5.1 Exportar Tabela MTI

In [ ]:
# Exportar Tabela MTI para LaTeX
mti_latex_file = f"{export_dir}/tabela_mti_llm_judge.tex"
mti_png_file = f"{export_dir}/tabela_mti_llm_judge.png"

# Configurar estilo para melhor visualização
df_mti_styled = df_mti_detailed.style.set_properties(**{
    'text-align': 'center',
    'font-size': '10pt',
    'border': '1px solid black'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4472C4'), ('color', 'white'), 
                                   ('font-weight', 'bold'), ('text-align', 'center'),
                                   ('border', '1px solid black')]},
    {'selector': 'td', 'props': [('border', '1px solid black')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]}
])

# Função para gerar LaTeX customizado
def generate_custom_latex(df, caption, label, approach_name):
    """Gera código LaTeX formatado"""
    
    # Cabeçalho do LaTeX
    latex_lines = []
    latex_lines.append("\\begin{table}[htbp]")
    latex_lines.append(f"\\caption{{{caption}}}")
    latex_lines.append(f"\\label{{{label}}}")
    latex_lines.append("\\centering")
    latex_lines.append("")
    latex_lines.append("\\begin{adjustbox}{max width=\\textwidth}")
    
    # Determinar número de colunas
    num_cols = len(df.columns)
    col_format = 'l' + 'c' * (num_cols - 1)
    
    latex_lines.append(f"\\begin{{tabular}}{{{col_format}}}")
    latex_lines.append("\\toprule")
    
    # Cabeçalho da tabela
    header_parts = []
    for col in df.columns:
        if col == "Modelo":
            header_parts.append("Modelo")
        else:
            header_parts.append(col)
    
    latex_lines.append(" & ".join(header_parts) + " \\\\")
    latex_lines.append("\\midrule")
    
    # Linhas de dados
    for idx, row in df.iterrows():
        row_parts = []
        for col in df.columns:
            cell_value = row[col]
            
            if col == "Modelo":
                # Nome do modelo
                row_parts.append(str(cell_value))
            else:
                # Processar valores de métrica (formato: "μ: 4.57 | Max: 57.71%")
                cell_str = str(cell_value)
                
                if "N/A" in cell_str:
                    row_parts.append("N/A")
                else:
                    # Parsear o formato "μ: 4.57 | Max: 57.71%"
                    if "|" in cell_str:
                        parts = cell_str.split("|")
                        mean_part = parts[0].strip()  # "μ: 4.57"
                        max_part = parts[1].strip()  # "Max: 57.71%"
                        
                        # Extrair valores numéricos
                        mean_value = mean_part.replace("μ:", "").strip()
                        max_value = max_part.replace("Max:", "").replace("%", "").strip()
                        
                        # Formatar no padrão LaTeX
                        latex_cell = f"$\\mu$: {mean_value} | Max: {max_value}\\%"
                        row_parts.append(latex_cell)
                    else:
                        row_parts.append(cell_str)
        
        latex_lines.append(" & ".join(row_parts) + " \\\\")
    
    # Rodapé da tabela
    latex_lines.append("\\bottomrule")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{adjustbox}")
    latex_lines.append("")
    latex_lines.append("\\end{table}")
    
    return "\n".join(latex_lines)

# Gerar LaTeX customizado para MTI
latex_output = generate_custom_latex(
    df_mti_detailed,
    caption="Resultados Detalhados da Abordagem MTI - Métricas LLM Judge por Modelo",
    label="tab:mti_llm_judge",
    approach_name="MTI"
)

# Salvar LaTeX
with open(mti_latex_file, 'w', encoding='utf-8') as f:
    f.write(latex_output)

# Exportar para PNG
try:
    dfi.export(df_mti_styled, mti_png_file, max_rows=-1, max_cols=-1)
    print(f"✅ Tabela MTI exportada com sucesso!")
except Exception as e:
    # Alternativa usando matplotlib
    print(f"⚠️ Usando método alternativo para PNG: {str(e)}")
    
    fig, ax = plt.subplots(figsize=(20, len(df_mti_detailed) * 0.5 + 2))
    ax.axis('tight')
    ax.axis('off')
    
    table = ax.table(cellText=df_mti_detailed.values, 
                     colLabels=df_mti_detailed.columns,
                     cellLoc='center', 
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Estilizar cabeçalho
    for i in range(len(df_mti_detailed.columns)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Estilizar células
    for i in range(1, len(df_mti_detailed) + 1):
        for j in range(len(df_mti_detailed.columns)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
    
    plt.savefig(mti_png_file, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.close()
    print(f"✅ Tabela MTI exportada com sucesso (método alternativo)!")

print(f"   📄 LaTeX: {mti_latex_file}")
print(f"   🖼️  PNG: {mti_png_file}")
print("-"*100)

### 5.2 Exportar Tabela STI

In [ ]:
# Exportar Tabela STI para LaTeX
sti_latex_file = f"{export_dir}/tabela_sti_llm_judge.tex"
sti_png_file = f"{export_dir}/tabela_sti_llm_judge.png"

# Configurar estilo
df_sti_styled = df_sti_detailed.style.set_properties(**{
    'text-align': 'center',
    'font-size': '10pt',
    'border': '1px solid black'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1A76FF'), ('color', 'white'), 
                                   ('font-weight', 'bold'), ('text-align', 'center'),
                                   ('border', '1px solid black')]},
    {'selector': 'td', 'props': [('border', '1px solid black')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]}
])

# Gerar LaTeX customizado para STI
latex_output = generate_custom_latex(
    df_sti_detailed,
    caption="Resultados Detalhados da Abordagem STI - Métricas LLM Judge por Modelo",
    label="tab:sti_llm_judge",
    approach_name="STI"
)

# Salvar LaTeX
with open(sti_latex_file, 'w', encoding='utf-8') as f:
    f.write(latex_output)

# Exportar para PNG
try:
    dfi.export(df_sti_styled, sti_png_file, max_rows=-1, max_cols=-1)
    print(f"✅ Tabela STI exportada com sucesso!")
except Exception as e:
    # Alternativa usando matplotlib
    print(f"⚠️ Usando método alternativo para PNG: {str(e)}")
    
    fig, ax = plt.subplots(figsize=(20, len(df_sti_detailed) * 0.5 + 2))
    ax.axis('tight')
    ax.axis('off')
    
    table = ax.table(cellText=df_sti_detailed.values, 
                     colLabels=df_sti_detailed.columns,
                     cellLoc='center', 
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Estilizar cabeçalho
    for i in range(len(df_sti_detailed.columns)):
        table[(0, i)].set_facecolor('#1A76FF')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Estilizar células
    for i in range(1, len(df_sti_detailed) + 1):
        for j in range(len(df_sti_detailed.columns)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
    
    plt.savefig(sti_png_file, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.close()
    print(f"✅ Tabela STI exportada com sucesso (método alternativo)!")

print(f"   📄 LaTeX: {sti_latex_file}")
print(f"   🖼️  PNG: {sti_png_file}")
print("-"*100)

### 5.3 Resumo da Exportação

In [ ]:
print("\n" + "="*100)
print("📦 RESUMO DA EXPORTAÇÃO")
print("="*100)

print(f"\n✅ Arquivos exportados com sucesso para o diretório: {export_dir}\n")

print("📊 TABELA MTI (Multi-Task Inference):")
print(f"   • Arquivo LaTeX: {mti_latex_file}")
print(f"   • Arquivo PNG: {mti_png_file}")

print("\n📊 TABELA STI (Single-Task Inference):")
print(f"   • Arquivo LaTeX: {sti_latex_file}")
print(f"   • Arquivo PNG: {sti_png_file}")

print("\n" + "="*100)
print("📝 INSTRUÇÕES DE USO NO LATEX")
print("="*100)

print("""
Para usar as tabelas no seu documento LaTeX:

1️⃣ OPÇÃO 1 - Usar arquivo .tex (Recomendado):
   
   No preâmbulo do seu documento, adicione:
   \\usepackage{booktabs}
   \\usepackage{adjustbox}
   
   No corpo do documento, onde deseja inserir a tabela:
   \\input{path/to/tabela_mti_llm_judge.tex}
   \\input{path/to/tabela_sti_llm_judge.tex}

2️⃣ FORMATO DA TABELA:
   • μ: [Média] - Pontuação média na escala Likert (1-5)
   • Max: [%] - Porcentagem de respostas com pontuação máxima (5)
   
3️⃣ DICAS:
   • As tabelas já incluem caption e label
   • Para formato paisagem: \\usepackage{rotating} e \\begin{sidewaystable}
   • Para ajustar fonte: \\small ou \\footnotesize antes da tabela
   • Referências: \\ref{tab:mti_llm_judge} ou \\ref{tab:sti_llm_judge}
""")

print("="*100)
print(f"🎉 Exportação concluída com sucesso!")
print("="*100)